In [1]:
try:
    import franken
except ImportError:
    %pip install "franken[mace]"
    import franken

try:
    import ipywidgets as widgets
except ImportError:
    %pip install ipywidgets
    import ipywidgets as widgets

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import clear_output, display
from pathlib import Path
from franken.metrics import is_pareto_efficient

## Analysing the results

In [2]:
## specify here your path to the run folder containing the log.json file
run_path = Path("/home/lbonati@iit.local/work/code/franken-new/experiments/results/run_260904_120758_aa4e9788/")

The two main outputs are the model trained with the selected hyperparameters, saved as `best_ckpt.pt`, and `log.json`, which describes every trained model. The analysis below only needs `log.json`. Select one or more validation metrics to minimize; the selected trial is recomputed interactively using autotune's Pareto-front and L1-norm selection rule.

In [3]:
# Load the logs for all trials.
with open(run_path / "log.json", "r") as fh:
    all_logs = json.load(fh)
logs_df = pd.json_normalize(all_logs)

available_selection_metrics = [
    metric
    for metric in all_logs[0]["metrics"]["validation"]
    if all(metric in trial["metrics"]["validation"] for trial in all_logs)
]
default_selection = tuple(
    metric
    for metric in ("energy_MAE", "forces_MAE")
    if metric in available_selection_metrics
) or (available_selection_metrics[0],)

best_model_selection = widgets.SelectMultiple(
    options=available_selection_metrics,
    value=default_selection,
    description="Minimize:",
    rows=min(6, len(available_selection_metrics)),
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px"),
)
best_model_output = widgets.Output()



def select_best_model(metrics_to_minimize, split="validation"):
    """Reproduce LogCollection.get_best_model using the loaded JSON logs."""
    if not metrics_to_minimize:
        raise ValueError("Select at least one metric to minimize.")

    costs = np.array(
        [
            [trial["metrics"][split][metric] for metric in metrics_to_minimize]
            for trial in all_logs
        ],
        dtype=float,
    )
    costs = np.nan_to_num(costs, nan=np.inf, posinf=np.inf, neginf=-np.inf)
    scores = np.linalg.norm(costs, ord=1, axis=1)
    scores[~is_pareto_efficient(costs)] = np.inf
    best_index = int(np.argmin(scores))
    return all_logs[best_index], best_index, scores[best_index]


def refresh_best_model(_=None):
    global best_log, best_trial_index, best_selection_score
    global best_checkpoint_hash, best_rf_weight_id
    global best_ls, best_l2, best_fw

    metrics = list(best_model_selection.value)
    with best_model_output:
        clear_output(wait=True)
        if not metrics:
            print("Select at least one validation metric to minimize.")
            return

        best_log, best_trial_index, best_selection_score = select_best_model(metrics)
        best_checkpoint_hash = best_log["checkpoint"]["hash"]
        best_rf_weight_id = best_log["checkpoint"]["rf_weight_id"]
        best_ls = best_log["hyperparameters"]["random_features"]["length_scale"]
        best_l2 = best_log["hyperparameters"]["solver"]["l2_penalty"]
        best_fw = best_log["hyperparameters"]["solver"]["forces_weight"]

        best_values = pd.DataFrame.from_dict(
            {
                "Length scale": best_ls,
                "L2 penalty": best_l2,
                "Force-to-energy weight ratio": best_fw,
            },
            orient="index",
            columns=["Value"],
        )
        best_values.index.name = "Hyperparameter"
        best_metrics = pd.DataFrame.from_dict(best_log["metrics"], orient="index")
        best_metrics.index.name = "Split"

        print(
            f"Selected trial {best_trial_index} | "
            f"L1 score ({', '.join(metrics)}): {best_selection_score:.6g}\n"
            f"Checkpoint hash: {best_checkpoint_hash}\n"
            f"RF weight ID: {best_rf_weight_id} "
            "(for checkpoints saved with all models)"
        )
        display(best_values, best_metrics)

    if "refresh_best_plot_controls" in globals():
        refresh_best_plot_controls()


best_model_selection.observe(refresh_best_model, names="value")
display(
    widgets.VBox(
        [
            widgets.HTML(
                "<b>Validation metrics used for model selection</b><br>"
                "Use Ctrl/Cmd-click to select more than one."
            ),
            best_model_selection,
        ]
    ),
    best_model_output,
)
refresh_best_model()

Output()

To make the analysis easier, convert the JSON logs to a pandas DataFrame.

With three hyperparameters, it is difficult to visualize their effects simultaneously. Use the controls below to choose the hyperparameter shown on the horizontal axis and fix the other two either at `Best` for the model-selection metrics chosen above or at any sampled value. Both validation errors are shown so that their trade-off remains visible.

In [6]:
HYPERPARAMETERS = {
    "Length scale": {
        "column": "hyperparameters.random_features.length_scale",
        "scale": "linear",
        "format": ".3g",
    },
    "L2 penalty": {
        "column": "hyperparameters.solver.l2_penalty",
        "scale": "log",
        "format": ".2e",
    },
    "Force-to-energy weight ratio": {
        "column": "hyperparameters.solver.forces_weight",
        "scale": "log",
        "format": ".3g",
    },
}
METRICS = {
    "Energy MAE [meV/atom]": "metrics.validation.energy_MAE",
    "Forces MAE [meV/Å]": "metrics.validation.forces_MAE",
}

parameter_values = {
    name: sorted(logs_df[settings["column"]].dropna().unique())
    for name, settings in HYPERPARAMETERS.items()
}
widget_style = {"description_width": "initial"}


def format_parameter_value(name, value):
    return format(value, HYPERPARAMETERS[name]["format"])


def current_best_value(name):
    return {
        "Length scale": best_ls,
        "L2 penalty": best_l2,
        "Force-to-energy weight ratio": best_fw,
    }[name]


def fixed_value_options(name):
    best = format_parameter_value(name, current_best_value(name))
    sampled = [
        (format_parameter_value(name, value), str(index))
        for index, value in enumerate(parameter_values[name])
    ]
    return [(f"Best ({best})", "best"), *sampled]


def selected_fixed_value(name, control):
    if control.value == "best":
        return current_best_value(name)
    return parameter_values[name][int(control.value)]


def select_trials(plotted_parameters, fixed_controls):
    selected = logs_df
    for name, control in fixed_controls.items():
        if name in plotted_parameters:
            continue
        column = HYPERPARAMETERS[name]["column"]
        value = selected_fixed_value(name, control)
        selected = selected[
            np.isclose(selected[column], value, rtol=1e-12, atol=0.0)
        ]
    return selected


def fixed_values_title(plotted_parameters, fixed_controls):
    labels = []
    for name, control in fixed_controls.items():
        if name in plotted_parameters:
            continue
        value = selected_fixed_value(name, control)
        label = f"{name} = {format_parameter_value(name, value)}"
        if control.value == "best":
            label += " (best)"
        labels.append(label)
    return "Fixed: " + ", ".join(labels)


axis_1d = widgets.Dropdown(
    options=list(HYPERPARAMETERS),
    value="Force-to-energy weight ratio",
    description="Horizontal axis:",
    style=widget_style,
)
fixed_1d = {
    name: widgets.Dropdown(
        options=fixed_value_options(name),
        value="best",
        description=f"Fix {name}:",
        style=widget_style,
    )
    for name in HYPERPARAMETERS
}
output_1d = widgets.Output()


def refresh_1d(_=None):
    x_name = axis_1d.value
    for name, control in fixed_1d.items():
        control.disabled = name == x_name

    with output_1d:
        clear_output(wait=True)
        selected = select_trials({x_name}, fixed_1d)
        x_column = HYPERPARAMETERS[x_name]["column"]
        metric_columns = list(METRICS.values())
        curve = (
            selected.groupby(x_column, as_index=False)[metric_columns]
            .mean()
            .sort_values(x_column)
        )
        if curve.empty:
            print("No trials match the selected fixed values.")
            return

        fig, ax = plt.subplots(figsize=(7, 4.5))
        energy_line = ax.plot(
            curve[x_column],
            curve[METRICS["Energy MAE [meV/atom]"]],
            marker="o",
            label="Energy MAE",
        )
        ax2 = ax.twinx()
        forces_line = ax2.plot(
            curve[x_column],
            curve[METRICS["Forces MAE [meV/Å]"]],
            marker="o",
            label="Forces MAE",
            color="tab:red",
        )
        ax.set_xscale(HYPERPARAMETERS[x_name]["scale"])
        lines = energy_line + forces_line
        ax.legend(lines, [line.get_label() for line in lines])
        ax.set_xlabel(x_name)
        ax.set_ylabel("Energy MAE [meV/atom]")
        ax2.set_ylabel("Forces MAE [meV/Å]")
        ax.set_title(fixed_values_title({x_name}, fixed_1d), fontsize=10)
        fig.tight_layout()
        plt.show()


def refresh_best_plot_controls():
    """Update every `Best` label and redraw plots after model selection."""
    control_groups = [fixed_1d]
    if "fixed_2d" in globals():
        control_groups.append(fixed_2d)
    for controls in control_groups:
        for name, control in controls.items():
            selected_option = control.value
            control.options = fixed_value_options(name)
            control.value = selected_option

    refresh_1d()
    if "refresh_2d" in globals():
        refresh_2d()


axis_1d.observe(refresh_1d, names="value")
for control in fixed_1d.values():
    control.observe(refresh_1d, names="value")

display(widgets.VBox([axis_1d, *fixed_1d.values()]), output_1d)
refresh_1d()

Output()

For a two-dimensional view, choose two distinct hyperparameters as the horizontal and vertical axes. Fix the remaining hyperparameter at its best or any sampled value, and select which validation metric to display. If the same hyperparameter is selected for both axes, the controls automatically swap them.

In [7]:
axis_2d_x = widgets.Dropdown(
    options=list(HYPERPARAMETERS),
    value="Length scale",
    description="Horizontal axis:",
    style=widget_style,
)
axis_2d_y = widgets.Dropdown(
    options=list(HYPERPARAMETERS),
    value="L2 penalty",
    description="Vertical axis:",
    style=widget_style,
)
metric_2d = widgets.Dropdown(
    options=list(METRICS),
    value="Forces MAE [meV/Å]",
    description="Color:",
    style=widget_style,
)
fixed_2d = {
    name: widgets.Dropdown(
        options=fixed_value_options(name),
        value="best",
        description=f"Fix {name}:",
        style=widget_style,
    )
    for name in HYPERPARAMETERS
}
output_2d = widgets.Output()
updating_2d_axes = False


def keep_2d_axes_distinct(change):
    global updating_2d_axes
    if updating_2d_axes or axis_2d_x.value != axis_2d_y.value:
        return
    updating_2d_axes = True
    other = axis_2d_y if change["owner"] is axis_2d_x else axis_2d_x
    other.value = change["old"]
    updating_2d_axes = False


def refresh_2d(_=None):
    x_name = axis_2d_x.value
    y_name = axis_2d_y.value
    plotted = {x_name, y_name}
    for name, control in fixed_2d.items():
        control.disabled = name in plotted

    with output_2d:
        clear_output(wait=True)
        selected = select_trials(plotted, fixed_2d)
        x_column = HYPERPARAMETERS[x_name]["column"]
        y_column = HYPERPARAMETERS[y_name]["column"]
        pivot = selected.pivot_table(
            index=y_column,
            columns=x_column,
            values=METRICS[metric_2d.value],
            aggfunc="mean",
        ).sort_index(axis=0).sort_index(axis=1)
        if pivot.empty:
            print("No trials match the selected fixed value.")
            return

        fig, ax = plt.subplots(figsize=(7, 5))
        image = ax.imshow(
            pivot.to_numpy(), cmap="viridis_r", aspect="auto", origin="lower"
        )
        colorbar = fig.colorbar(image, ax=ax)
        colorbar.set_label(metric_2d.value)
        ax.set_xticks(
            range(len(pivot.columns)),
            [format_parameter_value(x_name, value) for value in pivot.columns],
        )
        ax.set_yticks(
            range(len(pivot.index)),
            [format_parameter_value(y_name, value) for value in pivot.index],
        )
        ax.set_xlabel(x_name)
        ax.set_ylabel(y_name)
        ax.set_title(fixed_values_title(plotted, fixed_2d), fontsize=10)
        fig.tight_layout()
        plt.show()


axis_2d_x.observe(keep_2d_axes_distinct, names="value")
axis_2d_y.observe(keep_2d_axes_distinct, names="value")
axis_2d_x.observe(refresh_2d, names="value")
axis_2d_y.observe(refresh_2d, names="value")
metric_2d.observe(refresh_2d, names="value")
for control in fixed_2d.values():
    control.observe(refresh_2d, names="value")

controls_2d = widgets.VBox(
    [axis_2d_x, axis_2d_y, metric_2d, *fixed_2d.values()]
)
display(controls_2d, output_2d)
refresh_2d()

Output()